## Koreksi seleksi (winner's curse) untuk peta kesetiaan head-level — LOKAL, tanpa GPU

Lanjutan `09_tahap2_peta_kesetiaan_kaggle.ipynb` → finding 05. Angka "head terbaik"
di sana = **maksimum dari 1024 lokasi**, jadi pasti menggelembung karena bias seleksi.
Notebook ini menjalankan 3 cek koreksi, semuanya dari file yang sudah didownload di
`output/09_tahap2_peta_kesetiaan_kaggle/`:

1. **Seleksi-evaluasi terpisah antar template** (dari CSV): pilih head pakai 3 template,
   ukur di template ke-4 yang nggak ikut milih (4 fold, leave-one-template-out).
2. **Max-statistic permutation test** (dari npz): null distribution untuk
   "rho maksimum atas 1024 head" — p-value yang SUDAH memperhitungkan seleksi.
3. **Split-half sel**: pilih head di separuh sel acak, evaluasi di separuh sisanya (200x).

Bonus: cek **head lintas-tipe L11 H16 & L18 H14** sebagai lokasi TETAP (bukan hasil
seleksi per tipe). Hasil bersih → `notes/findings/06_...`.


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

DATA_DIR = "output/09_tahap2_peta_kesetiaan_kaggle"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join("notebooks", DATA_DIR)  # kalau dijalankan dari root repo
assert os.path.isdir(DATA_DIR), f"folder data tidak ketemu: {DATA_DIR}"

RANDOM_SEED = 42
N_PERM = 2000        # permutasi untuk max-statistic test
N_SPLIT = 200        # ulangan split-half

heads_npz = np.load(os.path.join(DATA_DIR, "emb_heads.npz"), allow_pickle=True)
heads_all = heads_npz["emb"]                      # [4, n_g, 32, 32, 128] fp16
GROUP_KEYS = [str(g) for g in heads_npz["group_keys"]]
group_real_dist = np.load(os.path.join(DATA_DIR, "group_real_dist.npy"))
peta = pd.read_csv(os.path.join(DATA_DIR, "peta_kesetiaan_full.csv"))

N_TEMPLATES, n_g, NUM_LAYERS, NUM_HEADS, HEAD_DIM = heads_all.shape
attr_types = np.array([gk.split(" :: ", 1)[0] for gk in GROUP_KEYS])
TYPES = sorted(set(attr_types.tolist()))
type_idx = {t: np.where(attr_types == t)[0] for t in TYPES}
print(f"{n_g} sel, {len(TYPES)} tipe, heads {NUM_LAYERS}x{NUM_HEADS}x{HEAD_DIM}, "
      f"{N_TEMPLATES} template, peta {peta.shape}")


### 0. Siapkan bahan per tipe

Per tipe: submatrix jarak-asli antar sel (buang sel yang punya pasangan NaN biar
permutasi bersih) + jarak cosine per head (Tmean) sebagai matriks [1024, n_pairs].


In [ ]:
def clean_subset(subset_idx):
    """Buang sel yang menyebabkan NaN di submatrix jarak-asli dalam-tipe."""
    sub = group_real_dist[np.ix_(subset_idx, subset_idx)]
    keep = np.ones(len(subset_idx), dtype=bool)
    while True:
        m = sub[np.ix_(keep, keep)]
        nan_per_cell = np.isnan(m).sum(axis=1)
        if nan_per_cell.max() == 0:
            break
        worst_local = int(np.argmax(nan_per_cell))
        keep_pos = np.where(keep)[0]
        keep[keep_pos[worst_local]] = False
    return subset_idx[keep]

def cosine_dist_per_head(subset_idx, t_tag="Tmean"):
    """[NUM_LAYERS*NUM_HEADS, n_pairs] jarak cosine antar sel subset, per head."""
    if t_tag == "Tmean":
        X = heads_all[:, subset_idx].astype(np.float32).mean(axis=0)   # [n, 32, 32, 128]
    else:
        X = heads_all[int(t_tag[1]), subset_idx].astype(np.float32)
    n = len(subset_idx)
    X = X.reshape(n, NUM_LAYERS * NUM_HEADS, HEAD_DIM)
    Xn = X / (np.linalg.norm(X, axis=2, keepdims=True) + 1e-8)
    sims = np.einsum("ahd,bhd->hab", Xn, Xn)                            # [1024, n, n]
    iu = np.triu_indices(n, k=1)
    return 1.0 - sims[:, iu[0], iu[1]]                                  # [1024, n_pairs]

def rank_rows(M):
    """rankdata per baris, vectorized (average ties ~ double argsort cukup di data kontinu)."""
    order = np.argsort(M, axis=-1)
    ranks = np.empty_like(order, dtype=np.float64)
    rng_ = np.arange(M.shape[-1], dtype=np.float64)
    np.put_along_axis(ranks, order, np.broadcast_to(rng_, M.shape).copy(), axis=-1)
    return ranks

def spearman_matrix(rep_ranked, real_vec):
    """Spearman antara tiap baris rep_ranked [K, P] dan real_vec [P] (di-rank di sini)."""
    r = rankdata(real_vec)
    r = r - r.mean()
    R = rep_ranked - rep_ranked.mean(axis=1, keepdims=True)
    num = R @ r
    den = np.sqrt((R ** 2).sum(axis=1) * (r ** 2).sum()) + 1e-12
    return num / den

bahan = {}
for ty in TYPES:
    idxs = clean_subset(type_idx[ty])
    dropped = len(type_idx[ty]) - len(idxs)
    sub_real = group_real_dist[np.ix_(idxs, idxs)]
    rep = cosine_dist_per_head(idxs, "Tmean")
    bahan[ty] = dict(idxs=idxs, real=sub_real, rep=rep, rep_ranked=rank_rows(rep))
    print(f"{ty}: {len(idxs)} sel (buang {dropped}), {rep.shape[1]} pasangan")


### Cek 1 — Seleksi-evaluasi terpisah antar template (dari CSV)

Pilih head terbaik pakai rata-rata rho 3 template → laporkan rho head itu di template
ke-4 yang TIDAK ikut memilih. Diulang 4 fold. Kalau rho held-out tetap dekat angka
Tmean → kenaikannya bukan keberuntungan seleksi.


In [ ]:
ph = peta[peta["component"] == "head"].copy()
ph["loc"] = ph["layer"].astype(str) + ":" + ph["head"].astype(str)
piv = {ty: ph[ph["attr_type"] == ty].pivot_table(index="loc", columns="template", values="rho")
       for ty in TYPES}

print("=" * 88)
rows1 = []
for ty in TYPES:
    P = piv[ty]
    naive_resid = peta[(peta["attr_type"] == ty) & (peta["component"] == "resid") &
                       (peta["template"] == "Tmean")]["rho"].max()
    tmean_max = float(P["Tmean"].max())
    held = []
    for t_out in range(4):
        train_cols = [f"T{t}" for t in range(4) if t != t_out]
        sel_loc = P[train_cols].mean(axis=1).idxmax()
        held.append(float(P.loc[sel_loc, f"T{t_out}"]))
    held = np.array(held)
    rows1.append(dict(attr_type=ty, resid_baseline=float(naive_resid),
                      tmean_max_inflated=tmean_max,
                      heldout_mean=float(held.mean()), heldout_min=float(held.min())))
    print(f"{ty:20s} residual={naive_resid:+.3f}  max-Tmean(inflated)={tmean_max:+.3f}  "
          f"held-out per fold: {np.round(held, 3)}  mean={held.mean():+.3f}")
cek1 = pd.DataFrame(rows1)


### Cek 2 — Max-statistic permutation test (koreksi seleksi penuh)

Null hypothesis: tidak ada hubungan embedding-survei SAMA SEKALI. Tiap permutasi:
acak label sel di matriks jarak-asli → hitung ulang **max rho atas semua 1024 head**
→ itu distribusi null untuk statistik "rho head terbaik". p = seberapa sering max
null >= max observasi. Ini otomatis memperhitungkan "milih juara dari 1024".


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
print("=" * 88)
rows2 = []
for ty in TYPES:
    b = bahan[ty]
    n = len(b["idxs"])
    iu = np.triu_indices(n, k=1)
    real_flat = b["real"][iu]
    obs = spearman_matrix(b["rep_ranked"], real_flat)
    obs_max = float(obs.max())
    null_max = np.empty(N_PERM)
    for k in range(N_PERM):
        perm = rng.permutation(n)
        null_max[k] = spearman_matrix(b["rep_ranked"], b["real"][np.ix_(perm, perm)][iu]).max()
    p_sel = float((null_max >= obs_max).mean())
    rows2.append(dict(attr_type=ty, obs_max=obs_max, null_max_mean=float(null_max.mean()),
                      null_max_p95=float(np.quantile(null_max, 0.95)), p_selection_corrected=p_sel))
    print(f"{ty:20s} max-rho obs={obs_max:+.3f} | null max: mean={null_max.mean():+.3f} "
          f"p95={np.quantile(null_max, 0.95):+.3f} | p(terkoreksi-seleksi)={p_sel:.4f}")
cek2 = pd.DataFrame(rows2)


### Cek 3 — Split-half sel: seberapa stabil pilihan head + rho held-out

200x: bagi sel 1 tipe jadi dua paruh acak → pilih head terbaik di paruh A → ukur rho
head itu di paruh B. Melaporkan distribusi rho held-out (angka yang JUJUR untuk
"kalau kupakai head ini di data baru, dapat berapa?").


In [ ]:
rng = np.random.default_rng(RANDOM_SEED + 1)
print("=" * 88)
rows3 = []
for ty in TYPES:
    b = bahan[ty]
    idxs = b["idxs"]; n = len(idxs)
    full_iu = np.triu_indices(n, k=1)
    pair_pos = {(i, j): k for k, (i, j) in enumerate(zip(full_iu[0], full_iu[1]))}
    held_rhos, chosen = [], []
    for s in range(N_SPLIT):
        perm = rng.permutation(n)
        half = n // 2
        A, B = np.sort(perm[:half]), np.sort(perm[half:])
        if len(A) < 5 or len(B) < 5:
            continue
        iuA = np.triu_indices(len(A), k=1); iuB = np.triu_indices(len(B), k=1)
        posA = np.array([pair_pos[(i, j)] for ii, i in enumerate(A) for j in A[ii+1:]])
        posB = np.array([pair_pos[(i, j)] for ii, i in enumerate(B) for j in B[ii+1:]])
        realA = b["real"][np.ix_(A, A)][iuA]; realB = b["real"][np.ix_(B, B)][iuB]
        rhoA = spearman_matrix(rank_rows(b["rep"][:, posA]), realA)
        best = int(np.argmax(rhoA))
        rhoB = spearman_matrix(rank_rows(b["rep"][best:best+1, posB]), realB)[0]
        held_rhos.append(float(rhoB)); chosen.append(best)
    held_rhos = np.array(held_rhos)
    top_choice = pd.Series(chosen).value_counts().head(3)
    lbl = ", ".join(f"L{c // NUM_HEADS} H{c % NUM_HEADS} ({v}x)" for c, v in top_choice.items())
    rows3.append(dict(attr_type=ty, heldout_median=float(np.median(held_rhos)),
                      heldout_q25=float(np.quantile(held_rhos, 0.25)),
                      heldout_q75=float(np.quantile(held_rhos, 0.75)),
                      head_tersering=lbl))
    print(f"{ty:20s} rho held-out: median={np.median(held_rhos):+.3f} "
          f"IQR=[{np.quantile(held_rhos, 0.25):+.3f}, {np.quantile(held_rhos, 0.75):+.3f}] | "
          f"head tersering: {lbl}")
cek3 = pd.DataFrame(rows3)


### Bonus — head lintas-tipe sebagai lokasi TETAP: L11 H16 dan L18 H14

Klaim "head umum" nggak kena winner's curse per-tipe kalau lokasinya DITETAPKAN duluan
lintas semua tipe. Di sini: rho per tipe di 2 head itu + permutation p per tipe
(dikali koreksi Bonferroni x1024 sebagai batas paling konservatif).


In [ ]:
FIXED = [(11, 16), (18, 14)]
rng = np.random.default_rng(RANDOM_SEED + 2)
print("=" * 88)
rows4 = []
for (L, H) in FIXED:
    flat = L * NUM_HEADS + H
    for ty in TYPES:
        b = bahan[ty]
        n = len(b["idxs"]); iu = np.triu_indices(n, k=1)
        real_flat = b["real"][iu]
        obs = float(spearman_matrix(b["rep_ranked"][flat:flat+1], real_flat)[0])
        null = np.empty(1000)
        for k in range(1000):
            perm = rng.permutation(n)
            null[k] = spearman_matrix(b["rep_ranked"][flat:flat+1],
                                      b["real"][np.ix_(perm, perm)][iu])[0]
        p_raw = float((null >= obs).mean())
        rows4.append(dict(head=f"L{L} H{H}", attr_type=ty, rho=obs, p_raw=p_raw,
                          p_bonf_1024=min(1.0, p_raw * 1024)))
        print(f"L{L} H{H}  {ty:20s} rho={obs:+.3f}  p_raw={p_raw:.4f}  "
              f"p x1024(konservatif)={min(1.0, p_raw * 1024):.3f}")
cek4 = pd.DataFrame(rows4)


In [ ]:
OUT = os.path.join(DATA_DIR, "koreksi_seleksi")
os.makedirs(OUT, exist_ok=True)
cek1.to_csv(os.path.join(OUT, "cek1_heldout_template.csv"), index=False)
cek2.to_csv(os.path.join(OUT, "cek2_maxstat_permutation.csv"), index=False)
cek3.to_csv(os.path.join(OUT, "cek3_splithalf.csv"), index=False)
cek4.to_csv(os.path.join(OUT, "cek4_fixed_heads.csv"), index=False)
print("Tersimpan di", OUT)


### Cara baca

- **Cek 1**: `heldout_mean` per tipe = angka yang boleh dikutip untuk "head terbaik"
  (bandingkan dengan `tmean_max_inflated` dan `resid_baseline`).
- **Cek 2**: `p_selection_corrected` < 0.05 = keunggulan head terbaik BUKAN keberuntungan
  memilih juara dari 1024. `null_max_p95` = berapa rho maksimum yang bisa dicapai noise
  murni — pembanding paling penting buat angka finding 05.
- **Cek 3**: median rho held-out = ekspektasi jujur performa head terpilih di data baru;
  `head_tersering` = apakah pilihannya konsisten (head yang sama kepilih terus) atau
  lotere (beda-beda tiap split).
- **Bonus**: kalau L11 H16 signifikan di banyak tipe SETELAH x1024 → klaim "head geometri
  demografis umum" kuat.

Hasil → `notes/findings/06_koreksi-seleksi-peta.md`.
